<a href="https://colab.research.google.com/github/smanjullee/MAURICYCLE/blob/main/MAURICYCLE_DASHBOARD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MAURICYCLE AI BIN DASHBOARD

### Bin Status Legend:
*   **GREEN** - EMPTY BIN
*   **ORANGE** - HALF BIN
*   **RED** - FULL BIN

In [1]:
# Install necessary libraries
!pip install gradio pandas folium geopy

In [2]:
import gradio as gr
import pandas as pd
import folium
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os

# --- Configuration and Data Loading ---

# Attempt to load the logo image. Adjust path if necessary.
LOGO_PATH = 'newlogo.jpeg'

# Load the Excel file
try:
    df_gps = pd.read_excel('gps.xlsx')

    print("Original unique Latitude values:", df_gps['Latitude'].unique())
    print("Original unique Longitude values:", df_gps['Longitude'].unique())

    # Clean Latitude column: remove leading "'- and convert to numeric
    df_gps['Latitude'] = df_gps['Latitude'].astype(str).str.replace("'-", '', regex=False)
    # Ensure Latitude and Longitude are numeric, coercing errors to NaN
    df_gps['Latitude'] = pd.to_numeric(df_gps['Latitude'], errors='coerce')
    df_gps['Longitude'] = pd.to_numeric(df_gps['Longitude'], errors='coerce')

    print("DataFrame head after to_numeric (before dropna):")
    display(df_gps.head())

    # Drop rows where coordinates could not be converted
    df_gps.dropna(subset=['Latitude', 'Longitude'], inplace=True)

    print("DataFrame head after dropna:")
    display(df_gps.head())

except FileNotFoundError:
    print("Error: gps.xlsx not found. Please upload it to your Colab environment.")
    df_gps = pd.DataFrame({'Place': [], 'Weight(Kg)': [], 'Latitude': [], 'Longitude': []}) # Create an empty DataFrame to avoid errors

# Initialize geolocator with a user_agent
geolocator = Nominatim(user_agent="mauricycle_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Cache for geocoded locations to avoid repeated API calls
location_cache = {}

def get_coordinates(place_name):
    if place_name in location_cache:
        return location_cache[place_name]

    # First, try to find coordinates in the df_gps itself if it's one of the bin places
    # This part was primarily for dropdowns (source/destination) that might not be in df_gps
    # For bin locations, we will use the pre-processed df_gps directly in generate_bin_map

    try:
        location = geocode(place_name + ", Mauritius") # Add Mauritius to improve accuracy
        if location:
            coords = (location.latitude, location.longitude)
            location_cache[place_name] = coords
            return coords
    except Exception as e:
        print(f"Error geocoding {place_name}: {e}")
    location_cache[place_name] = None # Cache failed attempts too
    return None

# Pre-defined list of towns/villages in Mauritius for the 'Source' dropdown
mauritius_towns = [
    "Port Louis", "Beau Bassin-Rose Hill", "Vacoas-Phoenix", "Curepipe",
    "Quatre Bornes", "Saint Pierre", "Centre de Flacq", "Mahebourg",
    "Goodlands", "Triolet", "Bel Air Rivière Sèche", "Grand Gaube"
]

# Get unique 'Place' names for the 'Destination' dropdown
destination_places = sorted(df_gps['Place'].unique().tolist())

Original unique Latitude values: [-20.1619 -20.3163 -20.2638 -20.2981 -20.2333 -20.4081 -20.1897 -20.0182
 -20.0384 -20.0576 -20.278  -20.4003 -20.1194 -20.1386]
Original unique Longitude values: [57.4989 57.5259 57.4791 57.4783 57.4661 57.7    57.7144 57.5802 57.6506
 57.5503 57.372  57.5967 57.4914 57.5301]
DataFrame head after to_numeric (before dropna):


,Device ID,Place,Latitude,Longitude,Timestamp (MUT),Weight (kg),Battery (%),Solar Input (W),Accuracy (m)
0,GPS_SOLAR_001,Port Louis (Capital),-20.1619,57.4989,2026-05-05T14:12:00+04:00,25,87,12.4,3.1
1,GPS_SOLAR_002,Curepipe,-20.3163,57.5259,2026-05-05T14:17:00+04:00,25,86,12.7,3.0
2,GPS_SOLAR_003,Quatre Bornes,-20.2638,57.4791,2026-05-05T14:22:00+04:00,20,86,13.1,2.9
3,GPS_SOLAR_004,Vacoas-Phoenix,-20.2981,57.4783,2026-05-05T14:27:00+04:00,15,85,13.5,3.2
4,GPS_SOLAR_005,Beau Bassin–Rose Hill,-20.2333,57.4661,2026-05-05T14:32:00+04:00,10,85,13.8,3.0


DataFrame head after dropna:


,Device ID,Place,Latitude,Longitude,Timestamp (MUT),Weight (kg),Battery (%),Solar Input (W),Accuracy (m)
0,GPS_SOLAR_001,Port Louis (Capital),-20.1619,57.4989,2026-05-05T14:12:00+04:00,25,87,12.4,3.1
1,GPS_SOLAR_002,Curepipe,-20.3163,57.5259,2026-05-05T14:17:00+04:00,25,86,12.7,3.0
2,GPS_SOLAR_003,Quatre Bornes,-20.2638,57.4791,2026-05-05T14:22:00+04:00,20,86,13.1,2.9
3,GPS_SOLAR_004,Vacoas-Phoenix,-20.2981,57.4783,2026-05-05T14:27:00+04:00,15,85,13.5,3.2
4,GPS_SOLAR_005,Beau Bassin–Rose Hill,-20.2333,57.4661,2026-05-05T14:32:00+04:00,10,85,13.8,3.0


In [3]:
print("Columns in df_gps:", df_gps.columns.tolist())
print(df_gps.info())

Columns in df_gps: ['Device ID', 'Place', 'Latitude', 'Longitude', 'Timestamp (MUT)', 'Weight (kg)', 'Battery (%)', 'Solar Input (W)', 'Accuracy (m)']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Device ID        14 non-null     object 
 1   Place            14 non-null     object 
 2   Latitude         14 non-null     float64
 3   Longitude        14 non-null     float64
 4   Timestamp (MUT)  14 non-null     object 
 5   Weight (kg)      14 non-null     int64  
 6   Battery (%)      14 non-null     int64  
 7   Solar Input (W)  14 non-null     float64
 8   Accuracy (m)     14 non-null     float64
dtypes: float64(4), int64(2), object(3)
memory usage: 1.1+ KB
None


In [4]:
import gradio as gr
import pandas as pd
import folium
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os

# --- Configuration and Data Loading ---

# Attempt to load the logo image. Adjust path if necessary.
LOGO_PATH = 'newlogo.jpeg'

# Load the Excel file
try:
    df_gps = pd.read_excel('gps.xlsx')

    print("Original unique Latitude values:", df_gps['Latitude'].unique())
    print("Original unique Longitude values:", df_gps['Longitude'].unique())

    # Clean Latitude column: remove leading "'- and convert to numeric
    df_gps['Latitude'] = df_gps['Latitude'].astype(str).str.replace("'-", '', regex=False)
    # Ensure Latitude and Longitude are numeric, coercing errors to NaN
    df_gps['Latitude'] = pd.to_numeric(df_gps['Latitude'], errors='coerce')
    df_gps['Longitude'] = pd.to_numeric(df_gps['Longitude'], errors='coerce')

    print("DataFrame head after to_numeric (before dropna):")
    display(df_gps.head())

    # Drop rows where coordinates could not be converted
    df_gps.dropna(subset=['Latitude', 'Longitude'], inplace=True)

    print("DataFrame head after dropna:")
    display(df_gps.head())

except FileNotFoundError:
    print("Error: gps.xlsx not found. Please upload it to your Colab environment.")
    df_gps = pd.DataFrame({'Place': [], 'Weight(Kg)': [], 'Latitude': [], 'Longitude': []}) # Create an empty DataFrame to avoid errors

# Initialize geolocator with a user_agent
geolocator = Nominatim(user_agent="mauricycle_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Cache for geocoded locations to avoid repeated API calls
location_cache = {}

def get_coordinates(place_name):
    if place_name in location_cache:
        return location_cache[place_name]

    # First, try to find coordinates in the df_gps itself if it's one of the bin places
    # This part was primarily for dropdowns (source/destination) that might not be in df_gps
    # For bin locations, we will use the pre-processed df_gps directly in generate_bin_map

    try:
        location = geocode(place_name + ", Mauritius") # Add Mauritius to improve accuracy
        if location:
            coords = (location.latitude, location.longitude)
            location_cache[place_name] = coords
            return coords
    except Exception as e:
        print(f"Error geocoding {place_name}: {e}")
    location_cache[place_name] = None # Cache failed attempts too
    return None

# Pre-defined list of towns/villages in Mauritius for the 'Source' dropdown
mauritius_towns = [
    "Port Louis", "Beau Bassin-Rose Hill", "Vacoas-Phoenix", "Curepipe",
    "Quatre Bornes", "Saint Pierre", "Centre de Flacq", "Mahebourg",
    "Goodlands", "Triolet", "Bel Air Rivière Sèche", "Grand Gaube"
]

# Get unique 'Place' names for the 'Destination' dropdown
destination_places = sorted(df_gps['Place'].unique().tolist())

# --- Function to generate the map ---
def generate_bin_map(source=None, destination=None):
    print(f"generate_bin_map called with source: {source}, destination: {destination}") # Added for debugging

    # Initialize a map centered on Mauritius
    m = folium.Map(location=[-20.3, 57.5], zoom_start=10) # Centered on Mauritius

    # Add bin markers using pre-existing coordinates in df_gps
    for index, row in df_gps.iterrows():
        place = row['Place']
        weight = row['Weight (kg)']
        lat = row['Latitude']
        lon = row['Longitude']

        # Only add marker if valid coordinates exist
        if pd.notna(lat) and pd.notna(lon):
            color = 'green'  # Empty bin
            if weight >= 12 and weight <= 13:
                color = 'orange' # Half bin
            elif weight > 23:
                color = 'red'    # Full bin

            folium.CircleMarker(
                location=[lat, lon],
                radius=8,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                tooltip=f"Place: {place}<br>Weight: {weight} Kg"
            ).add_to(m)

    # Add path if source and destination are selected
    if source and destination and source != destination:
        source_coords = get_coordinates(source) # Use geocoding for dropdown selected source
        destination_coords = get_coordinates(destination) # Use geocoding for dropdown selected destination

        if source_coords and destination_coords:
            folium.PolyLine(
                locations=[source_coords, destination_coords],
                color='red',
                weight=5,
                opacity=0.8,
                tooltip=f"Path from {source} to {destination}"
            ).add_to(m)

    # Get the HTML representation of the Folium map directly
    map_html = m._repr_html_()

    # Wrap the Folium HTML in a div with explicit styling to ensure it renders correctly in Gradio
    wrapped_map_html = f"<div style='height: 700px; width: 100%;'>{map_html}</div>"

    print(f"Generated map HTML content length: {len(wrapped_map_html)} characters.") # Added for debugging
    return wrapped_map_html

Original unique Latitude values: [-20.1619 -20.3163 -20.2638 -20.2981 -20.2333 -20.4081 -20.1897 -20.0182
 -20.0384 -20.0576 -20.278  -20.4003 -20.1194 -20.1386]
Original unique Longitude values: [57.4989 57.5259 57.4791 57.4783 57.4661 57.7    57.7144 57.5802 57.6506
 57.5503 57.372  57.5967 57.4914 57.5301]
DataFrame head after to_numeric (before dropna):


,Device ID,Place,Latitude,Longitude,Timestamp (MUT),Weight (kg),Battery (%),Solar Input (W),Accuracy (m)
0,GPS_SOLAR_001,Port Louis (Capital),-20.1619,57.4989,2026-05-05T14:12:00+04:00,25,87,12.4,3.1
1,GPS_SOLAR_002,Curepipe,-20.3163,57.5259,2026-05-05T14:17:00+04:00,25,86,12.7,3.0
2,GPS_SOLAR_003,Quatre Bornes,-20.2638,57.4791,2026-05-05T14:22:00+04:00,20,86,13.1,2.9
3,GPS_SOLAR_004,Vacoas-Phoenix,-20.2981,57.4783,2026-05-05T14:27:00+04:00,15,85,13.5,3.2
4,GPS_SOLAR_005,Beau Bassin–Rose Hill,-20.2333,57.4661,2026-05-05T14:32:00+04:00,10,85,13.8,3.0


DataFrame head after dropna:


,Device ID,Place,Latitude,Longitude,Timestamp (MUT),Weight (kg),Battery (%),Solar Input (W),Accuracy (m)
0,GPS_SOLAR_001,Port Louis (Capital),-20.1619,57.4989,2026-05-05T14:12:00+04:00,25,87,12.4,3.1
1,GPS_SOLAR_002,Curepipe,-20.3163,57.5259,2026-05-05T14:17:00+04:00,25,86,12.7,3.0
2,GPS_SOLAR_003,Quatre Bornes,-20.2638,57.4791,2026-05-05T14:22:00+04:00,20,86,13.1,2.9
3,GPS_SOLAR_004,Vacoas-Phoenix,-20.2981,57.4783,2026-05-05T14:27:00+04:00,15,85,13.5,3.2
4,GPS_SOLAR_005,Beau Bassin–Rose Hill,-20.2333,57.4661,2026-05-05T14:32:00+04:00,10,85,13.8,3.0


In [ ]:
# --- Gradio Interface Layout ---
with gr.Blocks(title="MAURICYCLE AI BIN DASHBOARD") as demo:
    # Display the logo if available
    if LOGO_PATH and os.path.exists(LOGO_PATH):
        try:
            gr.Image(LOGO_PATH, width=100, height=100, show_label=False, container=False)
        except Exception as e:
            gr.Markdown(f"<p style='color:red;'>Error loading logo: {e}</p>")
    elif LOGO_PATH:
        gr.Markdown(f"<p style='color:orange;'>Warning: Logo file '{LOGO_PATH}' not found. Please upload it to your Colab environment.</p>")

    gr.Markdown("# MAURICYCLE AI BIN DASHBOARD")
    gr.Markdown("### Bin Status Legend: GREEN - EMPTY BIN | ORANGE - HALF BIN | RED - FULL BIN")

    with gr.Row():
        source_dropdown = gr.Dropdown(
            label="Source",
            choices=mauritius_towns,
            value=None,
            interactive=True
        )
        destination_dropdown = gr.Dropdown(
            label="Destination",
            choices=destination_places,
            value=None,
            interactive=True
        )

    show_route_button = gr.Button("SHOW ROUTE")

    # Provide an initial HTML value with explicit styling for height/width
    map_output = gr.HTML(label="Bin Locations and Path", value="<div style='height: 700px; width: 100%;'>Loading map...</div>")

    # Initial map generation on load: pass None to prevent route drawing initially
    demo.load(generate_bin_map, inputs=[gr.State(None), gr.State(None)], outputs=map_output)

    # Update map when button is clicked
    show_route_button.click(
        generate_bin_map,
        inputs=[source_dropdown, destination_dropdown],
        outputs=map_output
    )

# Launch the Gradio app
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b7130d4190fcbc789c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


generate_bin_map called with source: None, destination: None
Generated map HTML content length: 17430 characters.
generate_bin_map called with source: None, destination: None
Generated map HTML content length: 17430 characters.
generate_bin_map called with source: Vacoas-Phoenix, destination: Curepipe
Generated map HTML content length: 18415 characters.
